In [ ]:
!pip install -qU vllm polars datasets pyarrow


In [ ]:
from vllm import LLM, SamplingParams
from datasets import load_dataset
import re
import os
import json
from tqdm.auto import tqdm


In [ ]:
dataset = load_dataset(
    "HuggingFaceTB/Countdown-Task-GOLD",
    data_files="all/train-00000-of-00001.parquet"
)["train"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
MODEL_ID = "Qwen/Qwen3-8B-AWQ"


In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer. Be extremely concise. Focus only on the successful logical path. Avoid any conversational fillers, self-corrections, or trial-and-error markers like 'wait', 'retry', or 'let's try again'."


In [ ]:
llm = LLM(
    quantization="awq",
    model=MODEL_ID,
    gpu_memory_utilization=0.9,
    max_model_len=1536,
    enforce_eager=True,
)

sampling_params = SamplingParams(
    temperature=0.7,
    presence_penalty=1.1,
    top_p=0.9,
    n=5,
    max_tokens=1024,
    stop=["</answer>"]
)


INFO 04-15 20:55:45 [utils.py:233] non-default args: {'max_model_len': 1536, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen3-8B-AWQ'}
INFO 04-15 20:55:46 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-15 20:55:46 [model.py:1678] Using max model len 1536
INFO 04-15 20:55:47 [awq_marlin.py:249] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 04-15 20:55:47 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 04-15 20:55:48 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-15 20:55:48 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-15 20:55:48 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-15 20:55:48 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 04-15 20:55:48 [compilation.py:290] Enabled custom fusions: norm_quant, act_quant
WARNING 04-15 20:55:49 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


In [ ]:
def verify_solution(numbers, target, model_output):
    try:
        if '<answer>' not in model_output:
            return None

        full_ans = model_output.split('<answer>')[-1].replace('</answer>', '').strip()

        if '=' not in full_ans:
            return None

        equation = full_ans.split('=')[0].strip()

        try:
            if abs(eval(equation) - target) > 1e-5:
                return None
        except ZeroDivisionError:
            return None

        used_numbers = [int(n) for n in re.findall(r'\d+', equation)]
        if sorted(used_numbers) != sorted(numbers):
            return None

        think_match = re.search(r'<think>(.*?)(?:</think>|$)', model_output, re.DOTALL)
        if think_match:
            think_content = think_match.group(1).lower()
            garbage_words = ['retry', 'start over', 'let\'s try again']
            if any(word in think_content for word in garbage_words):
                return None

        if not model_output.isascii():
            return None

        if '</answer>' not in model_output:
            model_output = model_output.strip() + '</answer>'

        return model_output
    except Exception:
        return None


In [ ]:
def format_qwen_prompt(query):
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"

start_id = 70000
prompts = [format_qwen_prompt(dataset[q]["prompt"][1]["content"]) for q in range(start_id, 76000) if dataset[q]["prompt"][1]["role"] == "user"]


In [ ]:
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)


Rendering prompts:   0%|          | 0/6000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/30000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

In [ ]:
best_results = {}

for i, output in enumerate(outputs):

    original_row = dataset[i + start_id]
    numbers = original_row['nums']
    target = original_row['target']

    task_id = f"{target}_{sorted(numbers)}"

    user_prompt = original_row["prompt"][1]["content"]

    for res in output.outputs:
        cleaned_text = verify_solution(numbers, target, res.text)

        if cleaned_text:
            text_len = len(cleaned_text)

            if task_id not in best_results or text_len < best_results[task_id][0]:
                best_results[task_id] = (text_len, {
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt},
                        {"role": "assistant", "content": cleaned_text}
                    ]
                })

final_sft_data = [v[1] for v in best_results.values()]

print(f"Всего задач прогнано: {len(outputs)}")
print(f"Уникальных задач с верным решением: {len(final_sft_data)}")

output_path = "dataset_70000_.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for entry in final_sft_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


Всего задач прогнано: 6000
Уникальных задач с верным решением: 4679
